# Week 4 Assignment — CIFAR-10: ANN vs CNN
**Name:** Srikar Reddy

---

**Objective:** Build and compare ANN and CNN models on the CIFAR-10 dataset

**Dataset:** CIFAR-10 — 60,000 color images (32×32) across 10 classes

**Models:**
- Baseline ANN (flattened input)
- CNN with Conv2D + BatchNorm + Pooling
- Advanced CNN with data augmentation

**Tasks:** Compare accuracy, train for 10/20 epochs, use EarlyStopping, analyze results

## Step 1: Imports

Setting a fixed random seed so the results stay consistent across runs.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping

np.random.seed(42)
tf.random.set_seed(42)

print("TF version:", tf.__version__)

TF version: 2.13.0


## Step 2: Load CIFAR-10

CIFAR-10 is built into Keras — 50,000 training images and 10,000 test images (32×32 RGB). Labels come as shape `(N,1)` so I flatten them to `(N,)` for `SparseCategoricalCrossentropy`.

In [2]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

y_train = y_train.flatten()
y_test  = y_test.flatten()

class_names = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']

print("Training set:", x_train.shape)
print("Test set    :", x_test.shape)
print("Pixel range :", x_train.min(), "-", x_train.max())

Training set: (50000, 32, 32, 3)
Test set    : (10000, 32, 32, 3)
Pixel range : 0 - 255


## Step 3: Visualize Sample Images

Quick look at the data before building any models. Images are only 32×32 which makes classification harder than it looks.

In [3]:
plt.figure(figsize=(12, 4))
for i in range(10):
    plt.subplot(2, 5, i+1)
    plt.imshow(x_train[i])
    plt.title(class_names[y_train[i]], fontsize=8)
    plt.axis('off')
plt.tight_layout()
plt.show()

<Figure size 1200x400 with 10 Axes>

## Step 4: Preprocessing

Dividing by 255.0 scales pixel values from 0-255 into [0,1], which keeps activations in a stable range during training. For the ANN, each 32×32×3 image also needs to be flattened into a 1D vector of 3072 values since Dense layers require flat input.

In [4]:
x_train_norm = x_train.astype('float32') / 255.0
x_test_norm  = x_test.astype('float32') / 255.0

# flatten for ANN  (32 * 32 * 3 = 3072)
x_train_flat = x_train_norm.reshape(len(x_train_norm), -1)
x_test_flat  = x_test_norm.reshape(len(x_test_norm), -1)

print("CNN input:", x_train_norm.shape)
print("ANN input:", x_train_flat.shape)

CNN input: (50000, 32, 32, 3)
ANN input: (50000, 3072)


## Step 5: Baseline ANN

The ANN flattens each image and processes it through Dense layers. It has no way of knowing which pixels are neighbors, so it loses all spatial structure — that's the main limitation we're trying to show here.

In [5]:
ann_model = models.Sequential([
    layers.Input(shape=(3072,)),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax')
], name='ANN_Baseline')

ann_model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
ann_model.summary()

Model: "ANN_Baseline"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 512)               1573376   
                                                                 
 dropout (Dropout)           (None, 512)               0         
                                                                 
 dense_1 (Dense)             (None, 256)               131328    
                                                                 
 dropout_1 (Dropout)         (None, 256)               0         
                                                                 
 dense_2 (Dense)             (None, 10)                2570      
                                                                 
Total params: 1707274 (6.51 MB)
Trainable params: 1707274 (6.51 MB)
Non-trainable params: 0 (0.00 B)
_________________________________________________________________


In [6]:
h_ann = ann_model.fit(x_train_flat, y_train,
                      epochs=10, batch_size=128,
                      validation_split=0.1, verbose=1)

ann_loss, ann_acc = ann_model.evaluate(x_test_flat, y_test, verbose=0)
print(f"\nANN Test Accuracy: {ann_acc:.4f}")

Epoch 1/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.2874 - loss: 2.0142 - val_accuracy: 0.3712 - val_loss: 1.7634
Epoch 2/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.3801 - loss: 1.7231 - val_accuracy: 0.4053 - val_loss: 1.6742
Epoch 3/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.4124 - loss: 1.6413 - val_accuracy: 0.4278 - val_loss: 1.6091
Epoch 4/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.4298 - loss: 1.5873 - val_accuracy: 0.4437 - val_loss: 1.5682
Epoch 5/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.4431 - loss: 1.5497 - val_accuracy: 0.4552 - val_loss: 1.5397
Epoch 6/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.4548 - loss: 1.5147 - val_accuracy: 0.4658 - val_loss: 1.5197
Epoch 7/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.4639 - loss: 1.4892 - val_accuracy: 0.4728 - val_loss: 1.5012
Epoch 8/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.4742 - loss: 1.4621 - val_accuracy: 0

## Step 6: Baseline CNN

The CNN uses convolution layers that slide filters across the image to detect local patterns like edges and textures, while preserving spatial structure. BatchNormalization stabilizes training and MaxPooling reduces the spatial size between blocks.

In [7]:
cnn_model = models.Sequential([
    layers.Input(shape=(32, 32, 3)),

    layers.Conv2D(32, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
], name='CNN_Baseline')

cnn_model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
cnn_model.summary()

Model: "CNN_Baseline"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 32, 32, 32)        896       
                                                                 
 batch_normalization (Batch  (None, 32, 32, 32)        128       
 Normalization)                                                  
                                                                 
 max_pooling2d (MaxPooling2  (None, 16, 16, 32)        0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 16, 16, 64)        18496     
                                                                 
 batch_normalization_1 (Bat  (None, 16, 16, 64)        256       
 chNormalization)                                                
                                                      

In [8]:
h_cnn = cnn_model.fit(x_train_norm, y_train,
                      epochs=10, batch_size=128,
                      validation_split=0.1, verbose=1)

cnn_loss, cnn_acc = cnn_model.evaluate(x_test_norm, y_test, verbose=0)
print(f"\nCNN Test Accuracy: {cnn_acc:.4f}")

Epoch 1/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 8s 20ms/step - accuracy: 0.3612 - loss: 1.7823 - val_accuracy: 0.5281 - val_loss: 1.3421
Epoch 2/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 7s 19ms/step - accuracy: 0.5537 - loss: 1.2691 - val_accuracy: 0.6014 - val_loss: 1.1284
Epoch 3/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 7s 19ms/step - accuracy: 0.6174 - loss: 1.0832 - val_accuracy: 0.6382 - val_loss: 1.0312
Epoch 4/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 7s 19ms/step - accuracy: 0.6527 - loss: 0.9897 - val_accuracy: 0.6641 - val_loss: 0.9671
Epoch 5/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 7s 19ms/step - accuracy: 0.6763 - loss: 0.9238 - val_accuracy: 0.6847 - val_loss: 0.9103
Epoch 6/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 7s 19ms/step - accuracy: 0.6943 - loss: 0.8741 - val_accuracy: 0.6978 - val_loss: 0.8751
Epoch 7/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 7s 19ms/step - accuracy: 0.7087 - loss: 0.8328 - val_accuracy: 0.7092 - val_loss: 0.8412
Epoch 8/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 7s 19ms/step - accuracy: 0.7195 - loss: 0.8011 - val_accu

## Step 7: ANN vs CNN — Learning Curves

Plotting both validation accuracy and validation loss side by side makes it easy to see not just which model is more accurate but also which one is overfitting.

In [9]:
plt.figure(figsize=(10, 4))

plt.subplot(1,2,1)
plt.plot(h_ann.history['val_accuracy'], label='ANN')
plt.plot(h_cnn.history['val_accuracy'], label='CNN')
plt.xlabel('Epoch'); plt.ylabel('Val Accuracy')
plt.title('Validation Accuracy'); plt.legend()

plt.subplot(1,2,2)
plt.plot(h_ann.history['val_loss'], label='ANN')
plt.plot(h_cnn.history['val_loss'], label='CNN')
plt.xlabel('Epoch'); plt.ylabel('Val Loss')
plt.title('Validation Loss'); plt.legend()

plt.tight_layout()
plt.show()

<Figure size 1000x400 with 2 Axes>

## Step 8: ANN with More Layers

Adding extra Dense layers with BatchNormalization to see if a deeper ANN can close the gap with CNN. Dropout rates are higher on larger layers to control overfitting.

In [10]:
ann_v2 = models.Sequential([
    layers.Input(shape=(3072,)),
    layers.Dense(1024, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(512, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.4),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax')
], name='ANN_v2')

ann_v2.compile(optimizer='adam',
               loss='sparse_categorical_crossentropy',
               metrics=['accuracy'])

h_ann2 = ann_v2.fit(x_train_flat, y_train,
                    epochs=10, batch_size=128,
                    validation_split=0.1, verbose=1)

_, ann_v2_acc = ann_v2.evaluate(x_test_flat, y_test, verbose=0)
print(f"\nANN v2 Test Accuracy: {ann_v2_acc:.4f}")

Epoch 1/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 6s 15ms/step - accuracy: 0.2631 - loss: 2.1083 - val_accuracy: 0.3814 - val_loss: 1.7291
Epoch 2/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - accuracy: 0.3792 - loss: 1.7204 - val_accuracy: 0.4148 - val_loss: 1.6347
Epoch 3/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - accuracy: 0.4163 - loss: 1.6218 - val_accuracy: 0.4389 - val_loss: 1.5692
Epoch 4/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - accuracy: 0.4389 - loss: 1.5614 - val_accuracy: 0.4551 - val_loss: 1.5218
Epoch 5/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - accuracy: 0.4537 - loss: 1.5178 - val_accuracy: 0.4672 - val_loss: 1.4887
Epoch 6/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - accuracy: 0.4663 - loss: 1.4821 - val_accuracy: 0.4793 - val_loss: 1.4601
Epoch 7/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - accuracy: 0.4763 - loss: 1.4531 - val_accuracy: 0.4862 - val_loss: 1.4434
Epoch 8/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - accuracy: 0.4851 - loss: 1.4287 - val_accu

## Step 9: CNN with 32→64→128 Filters

Adding a third conv block with 128 filters. Increasing filters at deeper layers is a common pattern — early layers detect simple edges, deeper layers combine them into more complex shapes.

In [11]:
cnn_v2 = models.Sequential([
    layers.Input(shape=(32, 32, 3)),

    layers.Conv2D(32, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(128, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
], name='CNN_v2')

cnn_v2.compile(optimizer='adam',
               loss='sparse_categorical_crossentropy',
               metrics=['accuracy'])

h_cnn2 = cnn_v2.fit(x_train_norm, y_train,
                    epochs=10, batch_size=128,
                    validation_split=0.1, verbose=1)

_, cnn_v2_acc = cnn_v2.evaluate(x_test_norm, y_test, verbose=0)
print(f"\nCNN v2 (32\u219264\u2192128) Test Accuracy: {cnn_v2_acc:.4f}")

Epoch 1/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 10s 26ms/step - accuracy: 0.3841 - loss: 1.7312 - val_accuracy: 0.5537 - val_loss: 1.2741
Epoch 2/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 9s 25ms/step - accuracy: 0.5763 - loss: 1.1981 - val_accuracy: 0.6284 - val_loss: 1.0612
Epoch 3/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 9s 25ms/step - accuracy: 0.6341 - loss: 1.0298 - val_accuracy: 0.6672 - val_loss: 0.9531
Epoch 4/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 9s 25ms/step - accuracy: 0.6714 - loss: 0.9387 - val_accuracy: 0.6883 - val_loss: 0.8972
Epoch 5/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 9s 25ms/step - accuracy: 0.6961 - loss: 0.8714 - val_accuracy: 0.7074 - val_loss: 0.8421
Epoch 6/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 9s 25ms/step - accuracy: 0.7162 - loss: 0.8192 - val_accuracy: 0.7198 - val_loss: 0.8097
Epoch 7/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 9s 25ms/step - accuracy: 0.7321 - loss: 0.7741 - val_accuracy: 0.7302 - val_loss: 0.7812
Epoch 8/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 9s 25ms/step - accuracy: 0.7449 - loss: 0.7384 - val_acc

## Step 10: CNN for 20 Epochs

Training the same baseline CNN for 20 epochs instead of 10 to check if extra training time improves accuracy or leads to overfitting.

In [12]:
cnn_20ep = models.Sequential([
    layers.Input(shape=(32, 32, 3)),

    layers.Conv2D(32, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
], name='CNN_20ep')

cnn_20ep.compile(optimizer='adam',
                 loss='sparse_categorical_crossentropy',
                 metrics=['accuracy'])

h_cnn_20 = cnn_20ep.fit(x_train_norm, y_train,
                        epochs=20, batch_size=128,
                        validation_split=0.1, verbose=1)

_, cnn_20ep_acc = cnn_20ep.evaluate(x_test_norm, y_test, verbose=0)
print(f"\nCNN (20 epochs) Test Accuracy: {cnn_20ep_acc:.4f}")

Epoch 1/20
352/352 ━━━━━━━━━━━━━━━━━━━━ 8s 20ms/step - accuracy: 0.3598 - loss: 1.7891 - val_accuracy: 0.5213 - val_loss: 1.3614
Epoch 2/20
352/352 ━━━━━━━━━━━━━━━━━━━━ 7s 19ms/step - accuracy: 0.5521 - loss: 1.2741 - val_accuracy: 0.5987 - val_loss: 1.1391
Epoch 3/20
352/352 ━━━━━━━━━━━━━━━━━━━━ 7s 19ms/step - accuracy: 0.6148 - loss: 1.0921 - val_accuracy: 0.6341 - val_loss: 1.0432
Epoch 4/20
352/352 ━━━━━━━━━━━━━━━━━━━━ 7s 19ms/step - accuracy: 0.6487 - loss: 0.9987 - val_accuracy: 0.6617 - val_loss: 0.9712
Epoch 5/20
352/352 ━━━━━━━━━━━━━━━━━━━━ 7s 19ms/step - accuracy: 0.6731 - loss: 0.9314 - val_accuracy: 0.6812 - val_loss: 0.9187
Epoch 6/20
352/352 ━━━━━━━━━━━━━━━━━━━━ 7s 19ms/step - accuracy: 0.6912 - loss: 0.8813 - val_accuracy: 0.6954 - val_loss: 0.8831
Epoch 7/20
352/352 ━━━━━━━━━━━━━━━━━━━━ 7s 19ms/step - accuracy: 0.7061 - loss: 0.8401 - val_accuracy: 0.7071 - val_loss: 0.8511
Epoch 8/20
352/352 ━━━━━━━━━━━━━━━━━━━━ 7s 19ms/step - accuracy: 0.7178 - loss: 0.8082 - val_accu

## Step 11: CNN with EarlyStopping

EarlyStopping monitors validation loss and stops training automatically when it stops improving. `patience=3` means it waits 3 more epochs before stopping, and `restore_best_weights=True` rolls back to the best checkpoint.

In [13]:
cnn_es = models.Sequential([
    layers.Input(shape=(32, 32, 3)),

    layers.Conv2D(32, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
], name='CNN_EarlyStopping')

cnn_es.compile(optimizer='adam',
               loss='sparse_categorical_crossentropy',
               metrics=['accuracy'])

es_callback = EarlyStopping(monitor='val_loss', patience=3,
                            restore_best_weights=True, verbose=1)

h_cnn_es = cnn_es.fit(x_train_norm, y_train,
                      epochs=30, batch_size=128,
                      validation_split=0.1,
                      callbacks=[es_callback], verbose=1)

_, cnn_es_acc = cnn_es.evaluate(x_test_norm, y_test, verbose=0)
print(f"\nCNN + EarlyStopping Test Accuracy: {cnn_es_acc:.4f}")
print(f"Stopped at epoch: {len(h_cnn_es.history['loss'])}")

Epoch 1/30
352/352 ━━━━━━━━━━━━━━━━━━━━ 8s 20ms/step - accuracy: 0.3621 - loss: 1.7812 - val_accuracy: 0.5247 - val_loss: 1.3521
Epoch 2/30
352/352 ━━━━━━━━━━━━━━━━━━━━ 7s 19ms/step - accuracy: 0.5548 - loss: 1.2614 - val_accuracy: 0.6021 - val_loss: 1.1241
Epoch 3/30
352/352 ━━━━━━━━━━━━━━━━━━━━ 7s 19ms/step - accuracy: 0.6181 - loss: 1.0814 - val_accuracy: 0.6398 - val_loss: 1.0284
Epoch 4/30
352/352 ━━━━━━━━━━━━━━━━━━━━ 7s 19ms/step - accuracy: 0.6531 - loss: 0.9871 - val_accuracy: 0.6652 - val_loss: 0.9641
Epoch 5/30
352/352 ━━━━━━━━━━━━━━━━━━━━ 7s 19ms/step - accuracy: 0.6772 - loss: 0.9212 - val_accuracy: 0.6854 - val_loss: 0.9071
Epoch 6/30
352/352 ━━━━━━━━━━━━━━━━━━━━ 7s 19ms/step - accuracy: 0.6954 - loss: 0.8724 - val_accuracy: 0.6994 - val_loss: 0.8721
Epoch 7/30
352/352 ━━━━━━━━━━━━━━━━━━━━ 7s 19ms/step - accuracy: 0.7098 - loss: 0.8311 - val_accuracy: 0.7108 - val_loss: 0.8387
Epoch 8/30
352/352 ━━━━━━━━━━━━━━━━━━━━ 7s 19ms/step - accuracy: 0.7211 - loss: 0.7994 - val_accu

## Step 12: CNN with Data Augmentation

Augmentation applies random flips, rotations and zooms to images during training so the model sees more varied inputs and generalizes better. These layers are part of the model so they automatically turn off during evaluation.

In [14]:
augment = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])

cnn_aug = models.Sequential([
    layers.Input(shape=(32, 32, 3)),
    augment,

    layers.Conv2D(32, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(128, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
], name='CNN_Augmented')

cnn_aug.compile(optimizer='adam',
                loss='sparse_categorical_crossentropy',
                metrics=['accuracy'])

h_cnn_aug = cnn_aug.fit(x_train_norm, y_train,
                        epochs=15, batch_size=128,
                        validation_split=0.1, verbose=1)

_, cnn_aug_acc = cnn_aug.evaluate(x_test_norm, y_test, verbose=0)
print(f"\nCNN + Augmentation Test Accuracy: {cnn_aug_acc:.4f}")

Epoch 1/15
352/352 ━━━━━━━━━━━━━━━━━━━━ 14s 36ms/step - accuracy: 0.3014 - loss: 1.9214 - val_accuracy: 0.4987 - val_loss: 1.4012
Epoch 2/15
352/352 ━━━━━━━━━━━━━━━━━━━━ 13s 35ms/step - accuracy: 0.4612 - loss: 1.5014 - val_accuracy: 0.5741 - val_loss: 1.1921
Epoch 3/15
352/352 ━━━━━━━━━━━━━━━━━━━━ 13s 35ms/step - accuracy: 0.5341 - loss: 1.3014 - val_accuracy: 0.6214 - val_loss: 1.0712
Epoch 4/15
352/352 ━━━━━━━━━━━━━━━━━━━━ 13s 35ms/step - accuracy: 0.5821 - loss: 1.1741 - val_accuracy: 0.6541 - val_loss: 0.9841
Epoch 5/15
352/352 ━━━━━━━━━━━━━━━━━━━━ 13s 35ms/step - accuracy: 0.6147 - loss: 1.0841 - val_accuracy: 0.6814 - val_loss: 0.9187
Epoch 6/15
352/352 ━━━━━━━━━━━━━━━━━━━━ 13s 35ms/step - accuracy: 0.6398 - loss: 1.0114 - val_accuracy: 0.7014 - val_loss: 0.8654
Epoch 7/15
352/352 ━━━━━━━━━━━━━━━━━━━━ 13s 35ms/step - accuracy: 0.6598 - loss: 0.9541 - val_accuracy: 0.7187 - val_loss: 0.8214
Epoch 8/15
352/352 ━━━━━━━━━━━━━━━━━━━━ 13s 35ms/step - accuracy: 0.6762 - loss: 0.9084 - 

## Step 13: Final Comparison

In [15]:
results = pd.DataFrame({
    'Model': [
        'ANN Baseline',
        'ANN with more layers',
        'CNN Baseline (32->64)',
        'CNN (32->64->128)',
        'CNN 20 Epochs',
        'CNN + EarlyStopping',
        'CNN + Data Augmentation'
    ],
    'Test Accuracy': [
        round(ann_acc, 4),
        round(ann_v2_acc, 4),
        round(cnn_acc, 4),
        round(cnn_v2_acc, 4),
        round(cnn_20ep_acc, 4),
        round(cnn_es_acc, 4),
        round(cnn_aug_acc, 4)
    ]
})
print(results.to_string(index=False))

                   Model  Test Accuracy
            ANN Baseline         0.5063
    ANN with more layers         0.5214
   CNN Baseline (32->64)         0.7187
       CNN (32->64->128)         0.7391
           CNN 20 Epochs         0.7298
     CNN + EarlyStopping         0.7241
 CNN + Data Augmentation         0.7612


In [16]:
plt.figure(figsize=(11, 5))
colors = ['#d9534f','#d9534f','#5b9bd5','#5b9bd5','#5b9bd5','#5b9bd5','#5b9bd5']
bars = plt.bar(results['Model'], results['Test Accuracy'], color=colors)

for bar, val in zip(bars, results['Test Accuracy']):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.004,
             str(val), ha='center', fontsize=8)

plt.xticks(rotation=30, ha='right', fontsize=8)
plt.ylim(0, 1.0)
plt.ylabel('Test Accuracy')
plt.title('All Models — CIFAR-10 Test Accuracy')
plt.tight_layout()
plt.show()

<Figure size 1100x500 with 1 Axes>

## Conclusion

- **ANN** plateaued around 50% — no matter how many layers you add, losing spatial structure at the flatten step is a fundamental bottleneck
- **Baseline CNN** jumped to ~72% in the same 10 epochs, proving that spatial feature extraction matters a lot for images
- **32→64→128 CNN** improved further by giving the model more capacity at deeper layers
- **20 epochs** helped slightly but validation accuracy started plateauing, showing diminishing returns
- **EarlyStopping** stopped at the optimal point and restored the best weights automatically
- **Data augmentation** gave the best generalization — the model was forced to learn more robust features instead of memorizing the training set